# RVC Whisper Encoder + LCNN ??? / ?? ???

??:
- ?? `best_model_rvc_v1.pt`? ?? LCNN/log-Mel ??? ???? ????, TTS ?? ??? ?? `Whisper base encoder + LCNN-style classifier` ??? RVC ?? ??? ?? ????.
- KANE, Nell_V2? ??/???? ???, Joonjong, NELL_KLM43x4? ??? ?? ?? RVC holdout?? ????.
- ??? Drive? `best_model_rvc_whisper_encoder_lcnn.pt`? `results/rvc_whisper_encoder_*` ??? ????.

In [ ]:
# 1. Colab ??
from google.colab import drive
drive.mount('/content/drive')

!pip -q install openai-whisper wandb

In [ ]:
# 2. ?? ??
from pathlib import Path
import os
import sys
import json
import time
import random
import shutil
import subprocess
from datetime import datetime

import torch

SEED = 42
random.seed(SEED)

DEEPVOICE_DIR = Path('/content/drive/MyDrive/deepvoice')
PROJECT_ROOT = DEEPVOICE_DIR / 'code' / 'v1'
DRIVE_FEATURE_DIR = DEEPVOICE_DIR / 'whisper_features'
LOCAL_RVC_FEATURE_DIR = Path('/content/deepvoice_rvc_whisper_features')

OLD_RVC_MODEL = DEEPVOICE_DIR / 'best_model_rvc_v1.pt'
OUTPUT_MODEL = DEEPVOICE_DIR / 'best_model_rvc_whisper_encoder_lcnn.pt'

RUN_PREPARE_SPLIT = True
RUN_TRAIN = True
RUN_EVALUATE_VAL = True
RUN_EVALUATE_HOLDOUT = True
RUN_EVALUATE_SPEAKERWISE = True
RUN_WANDB = True

# RVC ?? feature ?? ??. ?? Drive ???? ??? ??? ??.
RVC_TRAIN_SOURCE_DIRS = ['KANE', 'Nell_V2']
RVC_HOLDOUT_SOURCE_DIRS = ['Joonjong', 'NELL_KLM43x4']

# real ?? feature. final TTS?? ?? ?? real feature? ??? ?? ??.
REAL_TRAIN_SOURCE_CANDIDATES = ['real_sampled_aug_train_balanced', 'real_train']
REAL_VAL_SOURCE_CANDIDATES = ['real_val']
REAL_HOLDOUT_SOURCE_CANDIDATES = ['real_raw_holdout', 'real_val']

# KANE/Nell_V2 ?? 500? ??: 400 train + 100 val
RVC_TRAIN_PER_DIR = 400
RVC_VAL_PER_DIR = 100

EPOCHS = 10
BATCH_SIZE = 16       # A100?? 16 ??. OOM ?? 8? ???.
EVAL_BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
WEIGHT_DECAY = 1e-4

RVC_REAL_TRAIN_DIR = 'rvc_real_train_balanced'
RVC_REAL_VAL_DIR = 'rvc_real_val_balanced'
RVC_REAL_HOLDOUT_DIR = 'rvc_real_holdout'
RVC_FAKE_TRAIN_DIR = 'rvc_fake_train_kane_nell'
RVC_FAKE_VAL_DIR = 'rvc_fake_val_kane_nell'

RESULT_DIR = DEEPVOICE_DIR / 'results' / ('rvc_whisper_encoder_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print('DEEPVOICE_DIR          =', DEEPVOICE_DIR)
print('PROJECT_ROOT           =', PROJECT_ROOT)
print('DRIVE_FEATURE_DIR      =', DRIVE_FEATURE_DIR)
print('LOCAL_RVC_FEATURE_DIR  =', LOCAL_RVC_FEATURE_DIR)
print('OLD_RVC_MODEL          =', OLD_RVC_MODEL)
print('OUTPUT_MODEL           =', OUTPUT_MODEL)
print('RESULT_DIR             =', RESULT_DIR)
print('cuda available         =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu                    =', torch.cuda.get_device_name(0))

In [ ]:
# 3. ?? RVC checkpoint ??
print('old model exists:', OLD_RVC_MODEL.exists())
if OLD_RVC_MODEL.exists():
    ckpt = torch.load(OLD_RVC_MODEL, map_location='cpu')
    if isinstance(ckpt, dict):
        print('checkpoint keys:', list(ckpt.keys()))
        print('val_f1:', ckpt.get('val_f1'))
        print('threshold:', ckpt.get('threshold'))
        print('model_type:', ckpt.get('model_type'))
        print('config:', ckpt.get('config'))
    else:
        print('state_dict only checkpoint')
else:
    print('?? best_model_rvc_v1.pt? ????. ? Whisper encoder RVC ??? ?????.')

In [ ]:
# 4. Drive feature ?? ?? ??

def count_pts(base, name):
    p = Path(base) / name
    return len(list(p.glob('*.pt'))) if p.exists() else 0

print('=== ?? feature ?? ===')
check_names = sorted(set(
    RVC_TRAIN_SOURCE_DIRS + RVC_HOLDOUT_SOURCE_DIRS +
    REAL_TRAIN_SOURCE_CANDIDATES + REAL_VAL_SOURCE_CANDIDATES + REAL_HOLDOUT_SOURCE_CANDIDATES +
    ['fake_train', 'fake_val', 'elevenlabs_val', 'holdout_v2']
))
for name in check_names:
    print(f'{name:36s} {count_pts(DRIVE_FEATURE_DIR, name):7d}')

print('\n=== RVC? ??? ?? ?? ?? ===')
for p in sorted(DRIVE_FEATURE_DIR.iterdir()):
    if not p.is_dir():
        continue
    low = p.name.lower()
    if any(key.lower() in low for key in ['rvc', 'kane', 'nell', 'joon', 'klm', 'minjung']):
        print(f'{p.name:36s} {len(list(p.glob("*.pt"))):7d}')

In [ ]:
# 5. RVC ??/??/holdout split ???
# Drive ??? /content ?? ??? symlink?? ?? ??? ???.
# ?? ??? ???? ?? ?????.

def list_pts(base, name):
    p = Path(base) / name
    if not p.exists():
        return []
    return sorted(p.glob('*.pt'))

def first_existing_dir(candidates):
    for name in candidates:
        if count_pts(DRIVE_FEATURE_DIR, name) > 0:
            return name
    return None

def reset_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    for item in path.iterdir():
        if item.is_symlink() or item.is_file():
            item.unlink()
        elif item.is_dir():
            shutil.rmtree(item)

def link_files(files, dst_dir, prefix=''):
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    linked = 0
    for idx, src in enumerate(files):
        src = Path(src)
        dst = dst_dir / f'{prefix}{idx:06d}_{src.name}'
        if dst.exists() or dst.is_symlink():
            continue
        try:
            os.symlink(str(src), str(dst))
        except OSError:
            shutil.copy2(src, dst)
        linked += 1
    return linked

real_train_src = first_existing_dir(REAL_TRAIN_SOURCE_CANDIDATES)
real_val_src = first_existing_dir(REAL_VAL_SOURCE_CANDIDATES)
real_holdout_src = first_existing_dir(REAL_HOLDOUT_SOURCE_CANDIDATES)

print('real_train_src  =', real_train_src)
print('real_val_src    =', real_val_src)
print('real_holdout_src=', real_holdout_src)

missing_rvc = [name for name in RVC_TRAIN_SOURCE_DIRS + RVC_HOLDOUT_SOURCE_DIRS if count_pts(DRIVE_FEATURE_DIR, name) == 0]
if missing_rvc:
    raise FileNotFoundError(f'RVC feature ??? ?? ??? ????: {missing_rvc}. 4? ? ???? ?? ???? ??? ?????.')
if not real_train_src or not real_val_src or not real_holdout_src:
    raise FileNotFoundError('real ?? feature ??? ?? ?????.')

if RUN_PREPARE_SPLIT:
    LOCAL_RVC_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    for name in [RVC_REAL_TRAIN_DIR, RVC_REAL_VAL_DIR, RVC_REAL_HOLDOUT_DIR, RVC_FAKE_TRAIN_DIR, RVC_FAKE_VAL_DIR] + RVC_HOLDOUT_SOURCE_DIRS:
        reset_dir(LOCAL_RVC_FEATURE_DIR / name)

    fake_train_files = []
    fake_val_files = []
    split_info = {}
    for name in RVC_TRAIN_SOURCE_DIRS:
        files = list_pts(DRIVE_FEATURE_DIR, name)
        random.Random(SEED).shuffle(files)
        train_n = min(RVC_TRAIN_PER_DIR, max(1, int(len(files) * 0.8)))
        val_n = min(RVC_VAL_PER_DIR, max(1, len(files) - train_n))
        train_files = files[:train_n]
        val_files = files[train_n:train_n + val_n]
        fake_train_files.extend(train_files)
        fake_val_files.extend(val_files)
        split_info[name] = {'total': len(files), 'train': len(train_files), 'val': len(val_files)}

    real_train_files = list_pts(DRIVE_FEATURE_DIR, real_train_src)
    real_val_files = list_pts(DRIVE_FEATURE_DIR, real_val_src)
    real_holdout_files = list_pts(DRIVE_FEATURE_DIR, real_holdout_src)
    random.Random(SEED).shuffle(real_train_files)
    random.Random(SEED + 1).shuffle(real_val_files)

    real_train_pick = real_train_files[:len(fake_train_files)]
    real_val_pick = real_val_files[:len(fake_val_files)]

    if len(real_train_pick) < len(fake_train_files) or len(real_val_pick) < len(fake_val_files):
        raise ValueError('real train/val feature ?? RVC fake split?? ?????.')

    link_files(real_train_pick, LOCAL_RVC_FEATURE_DIR / RVC_REAL_TRAIN_DIR, 'realtrain_')
    link_files(real_val_pick, LOCAL_RVC_FEATURE_DIR / RVC_REAL_VAL_DIR, 'realval_')
    link_files(real_holdout_files, LOCAL_RVC_FEATURE_DIR / RVC_REAL_HOLDOUT_DIR, 'realholdout_')
    link_files(fake_train_files, LOCAL_RVC_FEATURE_DIR / RVC_FAKE_TRAIN_DIR, 'rvctrain_')
    link_files(fake_val_files, LOCAL_RVC_FEATURE_DIR / RVC_FAKE_VAL_DIR, 'rvcval_')

    for name in RVC_HOLDOUT_SOURCE_DIRS:
        link_files(list_pts(DRIVE_FEATURE_DIR, name), LOCAL_RVC_FEATURE_DIR / name, f'{name}_')

    manifest = {
        'seed': SEED,
        'real_train_src': real_train_src,
        'real_val_src': real_val_src,
        'real_holdout_src': real_holdout_src,
        'rvc_train_source_dirs': RVC_TRAIN_SOURCE_DIRS,
        'rvc_holdout_source_dirs': RVC_HOLDOUT_SOURCE_DIRS,
        'split_info': split_info,
        'counts': {},
    }
    for name in [RVC_REAL_TRAIN_DIR, RVC_REAL_VAL_DIR, RVC_REAL_HOLDOUT_DIR, RVC_FAKE_TRAIN_DIR, RVC_FAKE_VAL_DIR] + RVC_HOLDOUT_SOURCE_DIRS:
        manifest['counts'][name] = count_pts(LOCAL_RVC_FEATURE_DIR, name)
    (RESULT_DIR / 'rvc_split_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print('=== local split counts ===')
for name in [RVC_REAL_TRAIN_DIR, RVC_REAL_VAL_DIR, RVC_REAL_HOLDOUT_DIR, RVC_FAKE_TRAIN_DIR, RVC_FAKE_VAL_DIR] + RVC_HOLDOUT_SOURCE_DIRS:
    print(f'{name:36s} {count_pts(LOCAL_RVC_FEATURE_DIR, name):7d}')
print('manifest:', RESULT_DIR / 'rvc_split_manifest.json')

In [ ]:
# 6. W&B ??? ??
if RUN_WANDB:
    import wandb
    wandb.login()
else:
    print('RUN_WANDB=False')

In [ ]:
# 7. RVC Whisper encoder ?? ???

def run_cmd(cmd, log_name=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), text=True, capture_output=True)
    output = (proc.stdout or '') + ('\n' + proc.stderr if proc.stderr else '')
    print(output)
    if log_name:
        path = RESULT_DIR / log_name
        path.write_text('$ ' + ' '.join(cmd) + '\n\n' + output, encoding='utf-8')
        print('???:', path)
    if proc.returncode != 0:
        raise RuntimeError(f'command failed: {proc.returncode}')
    return output

train_cmd = [
    sys.executable,
    PROJECT_ROOT / 'model' / 'scripts' / 'train_whisper_encoder.py',
    '--deepvoice-dir', DEEPVOICE_DIR,
    '--data-dir', LOCAL_RVC_FEATURE_DIR,
    '--output-path', OUTPUT_MODEL,
    '--run-name', 'rvc_whisper_encoder_lcnn',
    '--real-train-dirs', RVC_REAL_TRAIN_DIR,
    '--real-val-dirs', RVC_REAL_VAL_DIR,
    '--fake-train-dirs', RVC_FAKE_TRAIN_DIR,
    '--fake-val-dirs', RVC_FAKE_VAL_DIR,
    '--epochs', EPOCHS,
    '--batch-size', BATCH_SIZE,
    '--num-workers', NUM_WORKERS,
    '--lr', LR,
    '--weight-decay', WEIGHT_DECAY,
    '--balanced-loss',
    '--freeze-whisper',
]
if RUN_WANDB:
    train_cmd += ['--wandb', '--wandb-project', 'deepvoice']

if RUN_TRAIN:
    run_cmd(train_cmd, 'train_rvc_whisper_encoder.txt')
else:
    print('RUN_TRAIN=False?? ??? ?????.')
    print('$', ' '.join(map(str, train_cmd)))

In [ ]:
# 8. ? RVC ?? checkpoint ??
print('output exists:', OUTPUT_MODEL.exists())
if OUTPUT_MODEL.exists():
    print('output size MB:', OUTPUT_MODEL.stat().st_size / 1024 / 1024)
    ckpt = torch.load(OUTPUT_MODEL, map_location='cpu')
    print('keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else 'state_dict only')
    if isinstance(ckpt, dict):
        print('model_type:', ckpt.get('model_type'))
        print('val_f1:', ckpt.get('val_f1'))
        print('threshold:', ckpt.get('threshold'))
        cfg = ckpt.get('config', {})
        for key in ['real_train_dirs', 'real_val_dirs', 'fake_train_dirs', 'fake_val_dirs', 'output_path']:
            print(f'{key}:', cfg.get(key))
else:
    print('?? ??? ???? ?????.')

In [ ]:
# 9. RVC val ??: KANE/Nell_V2?? ??? val split
val_cmd = [
    sys.executable,
    PROJECT_ROOT / 'model' / 'scripts' / 'evaluate_whisper_encoder.py',
    '--data-dir', LOCAL_RVC_FEATURE_DIR,
    '--model-path', OUTPUT_MODEL,
    '--real-dirs', RVC_REAL_VAL_DIR,
    '--fake-dirs', RVC_FAKE_VAL_DIR,
    '--batch-size', EVAL_BATCH_SIZE,
    '--num-workers', 0,
]

if RUN_EVALUATE_VAL:
    run_cmd(val_cmd, 'eval_rvc_val.txt')
else:
    print('RUN_EVALUATE_VAL=False')
    print('$', ' '.join(map(str, val_cmd)))

In [ ]:
# 10. RVC holdout ?? ??: Joonjong + NELL_KLM43x4
holdout_fake_dirs = ','.join(RVC_HOLDOUT_SOURCE_DIRS)
holdout_cmd = [
    sys.executable,
    PROJECT_ROOT / 'model' / 'scripts' / 'evaluate_whisper_encoder.py',
    '--data-dir', LOCAL_RVC_FEATURE_DIR,
    '--model-path', OUTPUT_MODEL,
    '--real-dirs', RVC_REAL_HOLDOUT_DIR,
    '--fake-dirs', holdout_fake_dirs,
    '--batch-size', EVAL_BATCH_SIZE,
    '--num-workers', 0,
]

if RUN_EVALUATE_HOLDOUT:
    run_cmd(holdout_cmd, 'eval_rvc_holdout_combined.txt')
else:
    print('RUN_EVALUATE_HOLDOUT=False')
    print('$', ' '.join(map(str, holdout_cmd)))

In [ ]:
# 11. RVC holdout ??/??? ?? ??
if RUN_EVALUATE_SPEAKERWISE:
    for name in RVC_HOLDOUT_SOURCE_DIRS:
        print('\n' + '=' * 100)
        print('RVC holdout:', name, 'count:', count_pts(LOCAL_RVC_FEATURE_DIR, name))
        cmd = [
            sys.executable,
            PROJECT_ROOT / 'model' / 'scripts' / 'evaluate_whisper_encoder.py',
            '--data-dir', LOCAL_RVC_FEATURE_DIR,
            '--model-path', OUTPUT_MODEL,
            '--real-dirs', RVC_REAL_HOLDOUT_DIR,
            '--fake-dirs', name,
            '--batch-size', EVAL_BATCH_SIZE,
            '--num-workers', 0,
        ]
        run_cmd(cmd, f'eval_rvc_holdout_{name}.txt')
else:
    print('RUN_EVALUATE_SPEAKERWISE=False')

In [ ]:
# 12. ?? ?? ?? ??
print('=== result files ===')
for p in sorted(RESULT_DIR.glob('*.txt')):
    print(p)

print('\n=== model ===')
print(OUTPUT_MODEL, OUTPUT_MODEL.exists())
if OUTPUT_MODEL.exists():
    print('size MB:', OUTPUT_MODEL.stat().st_size / 1024 / 1024)

## ?? ?? ??

- `eval_rvc_val.txt`: KANE/Nell_V2?? ??? validation ????. ?? ??? ?? RVC ????? ??? ?? ?? ? ??.
- `eval_rvc_holdout_combined.txt`: ??? ?? ?? Joonjong + NELL_KLM43x4 ?? holdout ????. ???? ? ? ??? ? ??? ? ????.
- `eval_rvc_holdout_*.txt`: RVC holdout ??? ????. ?? RVC ?? ????? ??? ????? ????.
- ??? ???? RVC? ?? ???? ?? ?? ?? ???? TTS ?? ??? ?? ??? ???? ?? ?? ????.